In [0]:

import json
from datetime import date
from urllib.parse import quote_plus
from pymongo import MongoClient
from common_utils.ingestor import read_cosmosdb_json, write_raw
from common_utils.logging import get_logger
     

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("cosmosdb-ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(f"{config_path}", "r") as f:
    config = json.load(f)

# ============================================================
# 3. CONFIG SECTIONS
# ============================================================
source_config = config["source"]
target_config = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

connection_string = dbutils.secrets.get(scope="retail-platform-dev", key=source_config["connection_string"])

# escape username/password in the mongodb URI (RFC 3986) — cosmos keys contain = + / characters that break URI parsing
scheme, _, rest = connection_string.partition("://")
userinfo, sep, host_part = rest.partition("@")
user, _, password = userinfo.partition(":")
if sep and password:
    connection_string = f"{scheme}://{quote_plus(user)}:{quote_plus(password)}{sep}{host_part}"

# ============================================================
# 5. READ FROM COSMOSDB SERVER
# ============================================================

df = read_cosmosdb_json(spark,connection_string,source_config["database_name"], source_config["collection_name"])
logger.info("Data Succesfully Read")
df.show()




# ============================================================
# 6. TARGET PATH
# ============================================================
target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"

logger.info("writiing data to %s", target_path)
# ============================================================
# 7. WRITE RAW
# ============================================================
logger.warning("Writing the in the target %s", target_path)
write_raw(df,target_path,target_config["file_format"],target_config["mode"])
logger.info("Data Succesfully written")



     